# 06 Baseline Model (Logistic Regression)
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Establish a clean, explainable, calibrated baseline classifier using Logistic Regression with stratified train/test split.


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

DATA_PROCESSED = "../data/processed"
df = pd.read_csv(os.path.join(DATA_PROCESSED, "attrition_features_engineered.csv"))

SENSITIVE_ATTRS = ['gender', 'marital_status']
TARGET_COLS = ['attrition', 'attrition_binary']
ID_COLS = ['employee_id']

feature_cols = [c for c in df.columns if c not in SENSITIVE_ATTRS + TARGET_COLS + ID_COLS]
X = df[feature_cols]
y = df['attrition_binary']

# Stratified split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")


Train size: 1176 | Test size: 294


In [3]:
# Preprocessing Pipeline
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)
y_probs = baseline_pipeline.predict_proba(X_test)[:, 1]

print("=== BASELINE LOGISTIC REGRESSION RESULTS ===")
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_probs), 4))
print("PR-AUC Score:", round(average_precision_score(y_test, y_probs), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


=== BASELINE LOGISTIC REGRESSION RESULTS ===
ROC-AUC Score: 0.7859
PR-AUC Score: 0.5387

Confusion Matrix:
[[197  50]
 [ 19  28]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.80      0.85       247
           1       0.36      0.60      0.45        47

    accuracy                           0.77       294
   macro avg       0.64      0.70      0.65       294
weighted avg       0.82      0.77      0.79       294

